In [3]:
# CELL — Left-merge 'Price Data' into 'raw_dump with Results dates' on (Ticker, Result Date)
import pandas as pd

WORKBOOK_PATH = "raw_indianapi_quarterly_final.xlsx"
RAW_SHEET     = "raw_dump with Results dates"
PRICE_SHEET   = "Price Data"
OUTPUT_SHEET  = "Merged_Data"

def _find_col_case_insensitive(df, target_name: str):
    tgt = target_name.strip().lower()
    for c in df.columns:
        if c.strip().lower() == tgt:
            return c
    raise SystemExit(f"Column '{target_name}' not found in sheet with columns: {list(df.columns)}")

def _norm_ticker(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.upper()

def _norm_result_date(s: pd.Series) -> pd.Series:
    # Try dayfirst first (your inputs look like dd-mm-yyyy), fallback to month-first, then keep as date()
    s1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
    s2 = pd.to_datetime(s, errors="coerce", dayfirst=False)
    s_final = s1.fillna(s2)
    return s_final.dt.date

# --- Load the two sheets (and only these two) ---
raw = pd.read_excel(WORKBOOK_PATH, sheet_name=RAW_SHEET)
price = pd.read_excel(WORKBOOK_PATH, sheet_name=PRICE_SHEET)

# Identify key columns (case-insensitive)
raw_ticker_col   = _find_col_case_insensitive(raw,   "Ticker")
raw_rdate_col    = _find_col_case_insensitive(raw,   "Result Date")
price_ticker_col = _find_col_case_insensitive(price, "Ticker")
price_rdate_col  = _find_col_case_insensitive(price, "Result Date")

# Build join keys (do NOT alter original columns)
raw["_k_ticker"]   = _norm_ticker(raw[raw_ticker_col])
raw["_k_rdate"]    = _norm_result_date(raw[raw_rdate_col])

price["_k_ticker"] = _norm_ticker(price[price_ticker_col])
price["_k_rdate"]  = _norm_result_date(price[price_rdate_col])

# To avoid row multiplication if Price Data has duplicates for a key, keep first
price_keys = ["_k_ticker", "_k_rdate"]
price_cols_to_add = [c for c in price.columns if c not in price_keys + [price_ticker_col, price_rdate_col]]
price_dedup = price.drop_duplicates(subset=price_keys, keep="first")[price_keys + price_cols_to_add]

# Left merge: keep ALL rows from RAW sheet
merged = raw.merge(price_dedup, on=price_keys, how="left")

# Drop helper key cols from output
merged = merged.drop(columns=price_keys)

# Write back to the SAME workbook, replacing/creating 'Merged_Data' only
with pd.ExcelWriter(WORKBOOK_PATH, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    merged.to_excel(xw, sheet_name=OUTPUT_SHEET, index=False)

print(f"Done. Wrote merged data to sheet '{OUTPUT_SHEET}' in {WORKBOOK_PATH}.")
print(f"Row count preserved: raw={len(raw)} -> merged={len(merged)}")


C:\Users\cecme\AppData\Local\Temp\ipykernel_54304\3757826720.py:22: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  s2 = pd.to_datetime(s, errors="coerce", dayfirst=False)


Done. Wrote merged data to sheet 'Merged_Data' in raw_indianapi_quarterly_final.xlsx.
Row count preserved: raw=585 -> merged=585
